In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Divvy-Bronze") \
    .getOrCreate()

print("Version Spark :", spark.version)
print("Version Python :", spark.sparkContext.pythonVer)

Version Spark : 4.3.0
Version Python : 3.13


In [2]:
try:
    import delta
    print("Delta Lake est installé.")
    print("Version :", delta.__version__)
except ImportError:
    print("Delta Lake n'est pas installé.")

Delta Lake n'est pas installé.


In [2]:
import sys
sys.path.append("/home/jovyan/work")

from commun.spark_utils import creer_session, RAW, LAKE
from commun.spark_utils import format_tables, vers_postgres

print("Import réussi !")
print("Dossier RAW :", RAW)
print("Data Lake :", LAKE)

Import réussi !
Dossier RAW : /home/jovyan/work/data/raw
Data Lake : s3a://datalake


In [3]:
spark = creer_session("Divvy-Bronze")
FMT = format_tables(spark)

print("Version Spark :", spark.version)
print("Format des tables :", FMT)

⚠ Delta Lake indisponible pour Spark 4.3.0 : les tables seront écrites en Parquet. Les démonstrations Delta (historique, time travel) seront sautées.
Préparation de Spark 4.3.0 (Hadoop 3.5.0) + data lake S3 … le premier démarrage peut prendre quelques minutes.


KeyboardInterrupt: 

In [4]:
print(spark.version)
print(spark.sparkContext.pythonVer)

NameError: name 'spark' is not defined

In [5]:
print("Version PySpark :", __import__("pyspark").__version__)

Version PySpark : 4.3.0


In [6]:
import sys
import pyspark

print("Python utilisé :", sys.executable)
print("PySpark installé ici :", pyspark.__file__)
print("Version :", pyspark.__version__)

Python utilisé : /opt/conda/bin/python
PySpark installé ici : /usr/local/spark/python/pyspark/__init__.py
Version : 4.3.0


In [7]:
import os
print("SPARK_HOME :", os.environ.get("SPARK_HOME"))

SPARK_HOME : /usr/local/spark


In [1]:
import pyspark
print(pyspark.__version__)

4.2.0


In [1]:
import sys
sys.path.append("/home/jovyan/work")

from commun.spark_utils import creer_session, format_tables

spark = creer_session("Divvy-Bronze")
FMT = format_tables(spark)

print("Spark :", spark.version)
print("Format :", FMT)

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040
Spark : 4.2.0
Format : delta


In [2]:
from commun.spark_utils import creer_bucket

creer_bucket()

Bucket « datalake » créé.


In [3]:
spark._jsc.hadoopConfiguration().get("fs.s3a.endpoint")

'http://rustfs:9000'

In [5]:
from commun.spark_utils import RAW

print("Dossier des données :", RAW)

Dossier des données : /home/jovyan/work/data/raw


In [6]:
from pyspark.sql import functions as F

divvy_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("recursiveFileLookup", "true") \
    .csv(RAW)

print("Nombre de lignes :", f"{divvy_raw.count():,}")
print("Nombre de colonnes :", len(divvy_raw.columns))
print("Colonnes :", divvy_raw.columns)

Nombre de lignes : 11,413,562
Nombre de colonnes : 13
Colonnes : ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']


In [8]:
from commun.spark_utils import LAKE

print("Data Lake :", LAKE)

Data Lake : s3a://datalake


In [9]:
divvy_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LAKE}/bronze/divvy_trips")

print("Couche Bronze enregistrée dans RustFS !")
print("Emplacement :", f"{LAKE}/bronze/divvy_trips")

Couche Bronze enregistrée dans RustFS !
Emplacement : s3a://datalake/bronze/divvy_trips


In [10]:
bronze = spark.read \
    .format("delta") \
    .load(f"{LAKE}/bronze/divvy_trips")

print("Nombre de lignes Bronze :", f"{bronze.count():,}")
print("Nombre de colonnes :", len(bronze.columns))

bronze.show(5, truncate=False)

Nombre de lignes Bronze : 11,413,562
Nombre de colonnes : 13
+----------------+-------------+-----------------------+-----------------------+---------------------------------+----------------+-----------------------------+--------------+------------------+------------------+---------+----------+-------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name               |start_station_id|end_station_name             |end_station_id|start_lat         |start_lng         |end_lat  |end_lng   |member_casual|
+----------------+-------------+-----------------------+-----------------------+---------------------------------+----------------+-----------------------------+--------------+------------------+------------------+---------+----------+-------------+
|D0086976F986F10C|classic_bike |2024-08-11 15:00:11.208|2024-08-11 15:15:56.321|Aberdeen St & Randolph St        |18062           |Daley Center Plaza           |TA1306000010  |41.884114    

In [1]:
from commun.spark_utils import creer_session, LAKE, format_tables

spark = creer_session("Divvy-Silver-Gold")
FMT = format_tables(spark)

print("Spark :", spark.version)
print("Format :", FMT)

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040
Spark : 4.2.0
Format : delta


In [2]:
bronze = spark.read \
    .format("delta") \
    .load(f"{LAKE}/bronze/divvy_trips")

print("Nombre de lignes Bronze :", f"{bronze.count():,}")

Nombre de lignes Bronze : 11,413,562


In [3]:
from pyspark.sql import functions as F

# 1. Vérifier les types des colonnes
bronze.printSchema()

# 2. Compter les valeurs nulles ou vides
colonnes = bronze.columns

diagnostic = bronze.agg(*[
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in colonnes
])

diagnostic.show(truncate=False)

# 3. Vérifier les catégories
print("Types de vélos :")
bronze.groupBy("rideable_type").count().show()

print("Types d'utilisateurs :")
bronze.groupBy("member_casual").count().show()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)

+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|
+-------+-------------+----------+--------+------------------+----------------+----------------+-----------

In [4]:
# Analyser les stations manquantes selon le type de véhicule

analyse_stations = bronze.groupBy("rideable_type").agg(
    F.count("*").alias("total_trajets"),

    F.sum(
        F.when(
            F.col("start_station_name").isNull() |
            (F.trim(F.col("start_station_name")) == ""),
            1
        ).otherwise(0)
    ).alias("sans_station_depart"),

    F.sum(
        F.when(
            F.col("end_station_name").isNull() |
            (F.trim(F.col("end_station_name")) == ""),
            1
        ).otherwise(0)
    ).alias("sans_station_arrivee")
)

analyse_stations.show(truncate=False)

+----------------+-------------+-------------------+--------------------+
|rideable_type   |total_trajets|sans_station_depart|sans_station_arrivee|
+----------------+-------------+-------------------+--------------------+
|electric_bike   |6585548      |2190975            |2264496             |
|classic_bike    |4683677      |0                  |13122               |
|electric_scooter|144337       |67649              |70340               |
+----------------+-------------+-------------------+--------------------+



In [5]:
# Vérification des dates
dates = bronze.select(
    F.to_timestamp("started_at").alias("debut"),
    F.to_timestamp("ended_at").alias("fin")
)

dates.select(
    F.sum(F.col("debut").isNull().cast("int")).alias("dates_debut_invalides"),
    F.sum(F.col("fin").isNull().cast("int")).alias("dates_fin_invalides")
).show()

# Vérification des identifiants dupliqués
print("Nombre de trajets :", bronze.count())
print("Nombre d'identifiants distincts :", bronze.select("ride_id").distinct().count())

+---------------------+-------------------+
|dates_debut_invalides|dates_fin_invalides|
+---------------------+-------------------+
|                    0|                  0|
+---------------------+-------------------+

Nombre de trajets : 11413562
Nombre d'identifiants distincts : 11413351


In [6]:
# Identifier les ride_id présents plusieurs fois
doublons = (
    bronze.groupBy("ride_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Nombre d'identifiants dupliqués :", doublons.count())
print(
    "Nombre de lignes excédentaires :",
    doublons.agg(
        F.sum(F.col("count") - 1)
    ).first()[0]
)

doublons.orderBy(F.desc("count")).show(10, truncate=False)

Nombre d'identifiants dupliqués : 211
Nombre de lignes excédentaires : 211
+----------------+-----+
|ride_id         |count|
+----------------+-----+
|5DCA3BA77BCB1B6D|2    |
|1EF6CBC15814F5DA|2    |
|997078F5B91EA984|2    |
|0FFF5607CF3F8A8D|2    |
|FA03ED5FBCE6E570|2    |
|CF25E56A43052CFF|2    |
|97FB4EDFD12E4192|2    |
|718E4ECBD0FE1440|2    |
|F8A9257D1DD04F43|2    |
|EDF25542931631EF|2    |
+----------------+-----+
only showing top 10 rows


In [8]:
analyse_durees = bronze.withColumn(
    "debut",
    F.to_timestamp("started_at")
).withColumn(
    "fin",
    F.to_timestamp("ended_at")
).withColumn(
    "duree_secondes",
    F.col("fin").cast("long") - F.col("debut").cast("long")
)

analyse_durees.select(
    F.min("duree_secondes").alias("duree_min_secondes"),
    F.max("duree_secondes").alias("duree_max_secondes"),
    F.sum(
        (F.col("duree_secondes") <= 0).cast("int")
    ).alias("durees_nulles_ou_negatives")
).show()

+------------------+------------------+--------------------------+
|duree_min_secondes|duree_max_secondes|durees_nulles_ou_negatives|
+------------------+------------------+--------------------------+
|           -164899|             94494|                       752|
+------------------+------------------+--------------------------+



In [9]:
analyse_durees.filter(
    F.col("duree_secondes") <= 0
).select(
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "duree_secondes",
    "member_casual"
).show(20, truncate=False)

+----------------+-------------+-----------------------+-----------------------+--------------+-------------+
|ride_id         |rideable_type|started_at             |ended_at               |duree_secondes|member_casual|
+----------------+-------------+-----------------------+-----------------------+--------------+-------------+
|A3C86101B94DCF83|classic_bike |2024-11-03 01:53:34.934|2024-11-03 01:04:55.434|-2919         |member       |
|9E3880C1B94EAB39|electric_bike|2024-11-03 01:48:07.988|2024-11-03 01:08:20.603|-2387         |casual       |
|A21F9A727675300F|electric_bike|2024-11-03 01:53:22.726|2024-11-03 01:06:40.133|-2802         |member       |
|61C2AE038C0D67AB|classic_bike |2024-11-03 01:25:33.243|2024-11-03 01:21:30.713|-243          |casual       |
|E44DD2F8C07D6B2D|electric_bike|2024-11-03 01:49:17.648|2024-11-03 01:02:39.700|-2798         |member       |
|F147973D700AA375|electric_bike|2024-11-03 01:50:02.506|2024-11-03 01:03:37.247|-2785         |casual       |
|F600DA96D

In [10]:
analyse_durees.filter(
    F.col("duree_secondes") > 86400
).select(
    "ride_id",
    "started_at",
    "ended_at",
    "duree_secondes",
    "member_casual"
).show(20, truncate=False)

+----------------+-----------------------+-----------------------+--------------+-------------+
|ride_id         |started_at             |ended_at               |duree_secondes|member_casual|
+----------------+-----------------------+-----------------------+--------------+-------------+
|AB15DC7A423435DF|2024-06-11 15:30:32.184|2024-06-12 16:30:17.256|89985         |casual       |
|100F27EC3569E83B|2024-06-14 20:52:01.904|2024-06-15 21:27:02.900|88501         |casual       |
|65B70111959D1213|2024-06-08 13:06:27.477|2024-06-09 13:58:31.566|89524         |casual       |
|DC66F5A5A4E3A460|2024-06-02 18:07:28.042|2024-06-03 18:31:22.024|87834         |member       |
|924768DB873351DE|2024-06-19 16:44:09.180|2024-06-20 16:50:42.010|86793         |casual       |
|AEB53F86DB9F7112|2024-06-10 10:13:20.814|2024-06-11 11:09:40.670|89780         |casual       |
|F1C86A24628FB248|2024-06-27 16:07:39.931|2024-06-28 17:03:19.106|89740         |casual       |
|753ADB37E1F60EB0|2024-06-09 11:14:14.44

In [11]:
analyse_durees.filter(
    F.col("duree_secondes") <= 0
).withColumn(
    "date_depart",
    F.to_date("started_at")
).groupBy(
    "date_depart"
).count().orderBy(
    F.desc("count")
).show(20, truncate=False)

+-----------+-----+
|date_depart|count|
+-----------+-----+
|2024-11-03 |43   |
|2025-11-02 |29   |
|2024-04-13 |24   |
|2024-05-04 |18   |
|2024-05-18 |18   |
|2024-05-25 |17   |
|2024-01-18 |16   |
|2024-05-22 |16   |
|2024-01-17 |16   |
|2024-04-14 |15   |
|2024-05-16 |14   |
|2024-05-17 |14   |
|2024-05-23 |12   |
|2024-05-03 |11   |
|2024-05-20 |11   |
|2024-05-01 |10   |
|2024-05-08 |10   |
|2024-05-21 |10   |
|2024-05-31 |10   |
|2024-04-08 |10   |
+-----------+-----+
only showing top 20 rows


In [12]:
analyse_durees.filter(
    (F.col("duree_secondes") <= 0) &
    (~F.to_date("started_at").isin(
        "2024-11-03",
        "2025-11-02"
    ))
).select(
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "duree_secondes",
    "member_casual"
).show(30, truncate=False)

+----------------+-------------+-------------------+-------------------+--------------+-------------+
|ride_id         |rideable_type|started_at         |ended_at           |duree_secondes|member_casual|
+----------------+-------------+-------------------+-------------------+--------------+-------------+
|DCE70CD2038926BD|classic_bike |2024-05-30 07:48:03|2024-05-30 07:48:03|0             |member       |
|56E3CFF914F39D77|electric_bike|2024-05-18 10:19:00|2024-05-18 10:19:00|0             |member       |
|3AC2A9E8EFCC39AD|electric_bike|2024-05-03 18:45:57|2024-05-03 18:45:55|-2            |member       |
|E8851B7313E96C6C|classic_bike |2024-05-04 16:04:10|2024-05-04 16:04:10|0             |member       |
|E55E54BF8B248375|electric_bike|2024-05-23 15:13:15|2024-05-23 15:09:39|-216          |member       |
|FD03DEBE335AD5A5|electric_bike|2024-05-31 23:38:44|2024-05-31 23:38:44|0             |casual       |
|9F2581BB31CCA9B7|electric_bike|2024-05-14 20:47:16|2024-05-14 20:47:16|0         

In [13]:
dates_changement = ["2024-11-03", "2025-11-02"]

# Nombre de trajets négatifs ou nuls hors changement d'heure
anomalies_hors_changement = analyse_durees.filter(
    (F.col("duree_secondes") <= 0) &
    (~F.to_date("started_at").isin(dates_changement))
).count()

# Nombre de trajets négatifs pendant les changements d'heure
anomalies_changement = analyse_durees.filter(
    (F.col("duree_secondes") < 0) &
    F.to_date("started_at").isin(dates_changement)
).count()

# Nombre de trajets exactement nuls
durees_nulles = analyse_durees.filter(
    F.col("duree_secondes") == 0
).count()

print("Durées nulles ou négatives hors changement d'heure :", anomalies_hors_changement)
print("Durées négatives pendant les changements d'heure :", anomalies_changement)
print("Durées exactement nulles :", durees_nulles)

Durées nulles ou négatives hors changement d'heure : 680
Durées négatives pendant les changements d'heure : 72
Durées exactement nulles : 496


In [14]:
from pyspark.sql import functions as F

dates_changement = ["2024-11-03", "2025-11-02"]

# 1. Conversion des types
silver = (
    bronze
    .withColumn("started_at", F.to_timestamp("started_at"))
    .withColumn("ended_at", F.to_timestamp("ended_at"))
    .withColumn("start_lat", F.col("start_lat").cast("double"))
    .withColumn("start_lng", F.col("start_lng").cast("double"))
    .withColumn("end_lat", F.col("end_lat").cast("double"))
    .withColumn("end_lng", F.col("end_lng").cast("double"))
)

# 2. Calcul de la durée
silver = silver.withColumn(
    "ride_duration_seconds",
    F.col("ended_at").cast("long") -
    F.col("started_at").cast("long")
)

# 3. Suppression des doublons
silver = silver.dropDuplicates(["ride_id"])

# 4. Suppression des durées invalides,
# sauf celles des deux changements d'heure
silver = silver.filter(
    (F.col("ride_duration_seconds") > 0) |
    (
        F.to_date("started_at").isin(dates_changement) &
        (F.col("ride_duration_seconds") < 0)
    )
)

# 5. Durée inconnue pour les trajets négatifs
# pendant les changements d'heure
silver = silver.withColumn(
    "ride_duration_seconds",
    F.when(
        F.col("ride_duration_seconds") < 0,
        None
    ).otherwise(F.col("ride_duration_seconds"))
)

# 6. Enregistrement dans RustFS
silver.write.format("delta") \
    .mode("overwrite") \
    .save(f"{LAKE}/silver/divvy_trips")

print("Couche Silver enregistrée avec succès.")

Couche Silver enregistrée avec succès.


In [15]:
silver = spark.read.format("delta").load(
    f"{LAKE}/silver/divvy_trips"
)

print("Nombre de lignes Bronze :", bronze.count())
print("Nombre de lignes Silver :", silver.count())
print("Nombre de lignes supprimées :", bronze.count() - silver.count())

silver.printSchema()

Nombre de lignes Bronze : 11413562
Nombre de lignes Silver : 11412671
Nombre de lignes supprimées : 891
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- ride_duration_seconds: long (nullable = true)



In [16]:
print("Identifiants dupliqués restants :",
      silver.count() - silver.select("ride_id").distinct().count())

print("Durées NULL :",
      silver.filter(
          F.col("ride_duration_seconds").isNull()
      ).count())

print("Stations de départ manquantes :",
      silver.filter(
          F.col("start_station_name").isNull()
      ).count())

print("Stations d'arrivée manquantes :",
      silver.filter(
          F.col("end_station_name").isNull()
      ).count())

Identifiants dupliqués restants : 0
Durées NULL : 72
Stations de départ manquantes : 2258232
Stations d'arrivée manquantes : 2347429


In [17]:
duplicate_ids = (
    bronze.groupBy("ride_id")
    .count()
    .filter(F.col("count") > 1)
    .select("ride_id")
)

duplicate_rows = bronze.join(
    duplicate_ids,
    on="ride_id",
    how="inner"
)

dup_check = duplicate_rows.groupBy("ride_id").agg(
    F.countDistinct(
        F.struct(*[F.col(c) for c in bronze.columns])
    ).alias("versions")
)

print(
    "Identifiants dupliqués avec des lignes différentes :",
    dup_check.filter(F.col("versions") > 1).count()
)

Identifiants dupliqués avec des lignes différentes : 211


In [18]:
duplicate_ids = (
    bronze.groupBy("ride_id")
    .count()
    .filter(F.col("count") > 1)
    .select("ride_id")
)

duplicate_rows = bronze.join(
    duplicate_ids,
    on="ride_id",
    how="inner"
)

# Afficher les 5 premiers identifiants concernés
sample_ids = [
    row["ride_id"]
    for row in duplicate_ids.limit(5).collect()
]

duplicate_rows.filter(
    F.col("ride_id").isin(sample_ids)
).orderBy("ride_id").show(20, truncate=False)

+----------------+-------------+-----------------------+-----------------------+---------------------------+----------------+-----------------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name         |start_station_id|end_station_name             |end_station_id|start_lat        |start_lng         |end_lat          |end_lng           |member_casual|
+----------------+-------------+-----------------------+-----------------------+---------------------------+----------------+-----------------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|29F756066AB38269|electric_bike|2024-05-31 23:55:26.801|2024-06-01 00:06:26.277|Dearborn Pkwy & Delaware Pl|TA1307000128    |NULL                         |NULL          |41.898926973     |-87.629926443     |41.93           

In [19]:
from pyspark.sql import functions as F

colonnes_hors_dates = [
    c for c in bronze.columns
    if c not in ["started_at", "ended_at"]
]

duplicates_normalises = (
    duplicate_rows
    .withColumn(
        "started_at_normalise",
        F.date_format(
            F.to_timestamp("started_at"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "ended_at_normalise",
        F.date_format(
            F.to_timestamp("ended_at"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
)

versions = duplicates_normalises.groupBy("ride_id").agg(
    F.countDistinct(
        F.struct(
            *[F.col(c) for c in colonnes_hors_dates],
            F.col("started_at_normalise"),
            F.col("ended_at_normalise")
        )
    ).alias("versions_normalisees")
)

print(
    "Doublons encore différents après normalisation :",
    versions.filter(
        F.col("versions_normalisees") > 1
    ).count()
)

Doublons encore différents après normalisation : 0


# 3. Couche Gold — Agrégation et préparation décisionnelle

## Objectif

La couche Gold transforme les données nettoyées de Silver
en tables agrégées destinées à répondre à la problématique :

« Dans quelles stations et à quelles périodes concentrer
les campagnes de conversion des utilisateurs occasionnels ? »

Deux tables sont construites :
- gold_station_period : analyse des trajets par station et période.
- gold_monthly_overview : évolution globale des trajets par mois.
- gold_station_ranking : classement des stations selon le volume de trajets occasionnels et leur part dans le trafic total. Un seuil minimal de 500 trajets sur la période étudiée est appliqué pour limiter les effets des faibles volumes.

In [1]:
print("Kernel opérationnel")

Kernel opérationnel


In [3]:
from commun.spark_utils import *

In [4]:
spark = creer_session("Divvy-Silver-Gold")

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040


In [5]:
print("Version Spark :", spark.version)

Version Spark : 4.2.0


In [6]:
silver = spark.read.format("delta").load(
    f"{LAKE}/silver/divvy_trips"
)

print("Couche Silver rechargée.")

Couche Silver rechargée.


In [7]:
spark.conf.set("spark.sql.session.timeZone", "America/Chicago")

print("Fuseau horaire :", spark.conf.get("spark.sql.session.timeZone"))

Fuseau horaire : America/Chicago


In [8]:
from pyspark.sql import functions as F

trajets_stations = silver.filter(
    F.col("start_station_id").isNotNull() &
    (F.trim(F.col("start_station_id")) != "")
)

print("Filtre appliqué : stations de départ renseignées.")

Filtre appliqué : stations de départ renseignées.


In [20]:
trajets_stations = (
    silver
    .filter(
        F.year("started_at").between(2024, 2025)
    )
    .filter(
        F.col("start_station_id").isNotNull() &
        (F.trim(F.col("start_station_id")) != "")
    )
    .withColumn("year", F.year("started_at"))
    .withColumn("month", F.month("started_at"))
    .withColumn("day_of_week", F.dayofweek("started_at"))
    .withColumn("hour", F.hour("started_at"))
)

In [21]:
gold_station_period = (
    trajets_stations
    .groupBy(
        "start_station_id",
        "start_station_name",
        "year",
        "month",
        "day_of_week",
        "hour",
        "rideable_type"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.sum(
            F.when(F.col("member_casual") == "casual", 1).otherwise(0)
        ).alias("casual_trips"),
        F.sum(
            F.when(F.col("member_casual") == "member", 1).otherwise(0)
        ).alias("member_trips")
    )
    .withColumn(
        "casual_share_pct",
        F.round(
            F.col("casual_trips") / F.col("total_trips") * 100,
            2
        )
    )
)

In [11]:
gold_station_period.orderBy(
    F.col("total_trips").desc()
).show(10, truncate=False)

+----------------+----------------------------------+----+-----+-----------+----+-------------+-----------+------------+------------+----------------+
|start_station_id|start_station_name                |year|month|day_of_week|hour|rideable_type|total_trips|casual_trips|member_trips|casual_share_pct|
+----------------+----------------------------------+----+-----+-----------+----+-------------+-----------+------------+------------+----------------+
|LF-005          |DuSable Lake Shore Dr & North Blvd|2024|8    |7          |14  |classic_bike |281        |207         |74          |73.67           |
|LF-005          |DuSable Lake Shore Dr & North Blvd|2024|7    |7          |16  |classic_bike |272        |191         |81          |70.22           |
|13022           |Streeter Dr & Grand Ave           |2024|5    |7          |13  |classic_bike |269        |219         |50          |81.41           |
|LF-005          |DuSable Lake Shore Dr & North Blvd|2024|7    |7          |15  |classic_bike 

In [12]:
controle_gold = gold_station_period.filter(
    (F.col("total_trips") != F.col("casual_trips") + F.col("member_trips")) |
    (F.col("casual_share_pct") < 0) |
    (F.col("casual_share_pct") > 100)
)

print("Nombre d'agrégations incohérentes :", controle_gold.count())

Nombre d'agrégations incohérentes : 0


In [13]:
gold_station_period.write.format("delta") \
    .mode("overwrite") \
    .save(f"{LAKE}/gold/station_period")

print("Table gold_station_period enregistrée avec succès.")

Table gold_station_period enregistrée avec succès.


In [16]:
gold_monthly_overview.orderBy(
    "year",
    "month",
    "member_casual",
    "rideable_type"
).show(20, truncate=False)

+----+-----+-------------+-------------+-----------+--------------------+--------------------+
|year|month|member_casual|rideable_type|total_trips|avg_duration_seconds|avg_duration_minutes|
+----+-----+-------------+-------------+-----------+--------------------+--------------------+
|2023|12   |casual       |classic_bike |27         |14204.666666666666  |236.74              |
|2023|12   |casual       |electric_bike|62         |681.3225806451613   |11.36               |
|2023|12   |member       |classic_bike |35         |718.5142857142857   |11.98               |
|2023|12   |member       |electric_bike|56         |817.0892857142857   |13.62               |
|2024|1    |casual       |classic_bike |10427      |2217.20015344778    |36.95               |
|2024|1    |casual       |electric_bike|13938      |554.4006313674846   |9.24                |
|2024|1    |member       |classic_bike |66037      |1030.496236958069   |17.17               |
|2024|1    |member       |electric_bike|54223     

In [17]:
silver.groupBy(
    F.year("started_at").alias("year")
).count().orderBy("year").show()

+----+-------+
|year|  count|
+----+-------+
|2023|    180|
|2024|5859886|
|2025|5552605|
+----+-------+



In [18]:
trajets_mensuels = (
    silver
    .filter(
        F.year("started_at").between(2024, 2025)
    )
    .withColumn("year", F.year("started_at"))
    .withColumn("month", F.month("started_at"))
)

In [19]:
gold_monthly_overview = (
    trajets_mensuels
    .groupBy(
        "year",
        "month",
        "member_casual",
        "rideable_type"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.avg("ride_duration_seconds").alias("avg_duration_seconds")
    )
    .withColumn(
        "avg_duration_minutes",
        F.round(F.col("avg_duration_seconds") / 60, 2)
    )
)

In [22]:
print("Années dans gold_station_period :")
gold_station_period.select("year").distinct().orderBy("year").show()

print("Années dans gold_monthly_overview :")
gold_monthly_overview.select("year").distinct().orderBy("year").show()

Années dans gold_station_period :
+----+
|year|
+----+
|2024|
|2025|
+----+

Années dans gold_monthly_overview :
+----+
|year|
+----+
|2024|
|2025|
+----+



In [23]:
gold_station_period.write.format("delta") \
    .mode("overwrite") \
    .save(f"{LAKE}/gold/station_period")

print("gold_station_period enregistrée.")

gold_station_period enregistrée.


In [24]:
gold_monthly_overview.write.format("delta") \
    .mode("overwrite") \
    .save(f"{LAKE}/gold/monthly_overview")

print("gold_monthly_overview enregistrée.")

gold_monthly_overview enregistrée.


In [25]:
gold_station_ranking = (
    trajets_stations
    .groupBy(
        "start_station_id",
        "start_station_name"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.sum(
            F.when(F.col("member_casual") == "casual", 1).otherwise(0)
        ).alias("casual_trips"),
        F.sum(
            F.when(F.col("member_casual") == "member", 1).otherwise(0)
        ).alias("member_trips")
    )
    .withColumn(
        "casual_share_pct",
        F.round(
            F.col("casual_trips") / F.col("total_trips") * 100,
            2
        )
    )
)

In [30]:
from pyspark.sql.window import Window

gold_station_ranking = (
    gold_station_ranking
    .withColumn(
        "rank_casual_volume",
        F.dense_rank().over(
            Window.orderBy(F.col("casual_trips").desc())
        )
    )
    .withColumn(
        "rank_casual_share",
        F.dense_rank().over(
            Window.orderBy(F.col("casual_share_pct").desc())
        )
    )
)

In [27]:
gold_station_ranking.orderBy(
    F.col("rank_casual_volume")
).select(
    "start_station_id",
    "start_station_name",
    "total_trips",
    "casual_trips",
    "member_trips",
    "casual_share_pct",
    "rank_casual_volume",
    "rank_casual_share"
).show(10, truncate=False)

+----------------+----------------------------------+-----------+------------+------------+----------------+------------------+-----------------+
|start_station_id|start_station_name                |total_trips|casual_trips|member_trips|casual_share_pct|rank_casual_volume|rank_casual_share|
+----------------+----------------------------------+-----------+------------+------------+----------------+------------------+-----------------+
|13022           |Streeter Dr & Grand Ave           |77318      |59513       |17805       |76.97           |1                 |153              |
|13300           |DuSable Lake Shore Dr & Monroe St |52493      |40829       |11664       |77.78           |2                 |142              |
|13042           |Michigan Ave & Oak St             |46293      |28851       |17442       |62.32           |3                 |371              |
|CHI01747        |Navy Pier                         |35135      |27414       |7721        |78.02           |4               

In [28]:
gold_station_ranking.orderBy(
    F.col("rank_casual_share")
).select(
    "start_station_id",
    "start_station_name",
    "total_trips",
    "casual_trips",
    "casual_share_pct",
    "rank_casual_share"
).show(10, truncate=False)

+----------------+--------------------------------------+-----------+------------+----------------+-----------------+
|start_station_id|start_station_name                    |total_trips|casual_trips|casual_share_pct|rank_casual_share|
+----------------+--------------------------------------+-----------+------------+----------------+-----------------+
|CHI01344        |Public Rack - Racine Ave & 109th Pl   |2          |2           |100.0           |1                |
|CHI02018        |Eggleston Ave & 115th St              |19         |19          |100.0           |1                |
|CHI00841        |Torrence Ave & 126th Pl               |31         |31          |100.0           |1                |
|CHI01710        |Public Rack - Western & 74th          |2          |2           |100.0           |1                |
|CHI01946        |State St & 127th St                   |20         |20          |100.0           |1                |
|CHI01393        |Public Rack - Michigan Ave & 119th St 

In [29]:
gold_station_ranking = (
    gold_station_ranking
    .filter(F.col("total_trips") >= 500)
    .drop("rank_casual_volume", "rank_casual_share")
)

In [31]:
gold_station_ranking.orderBy(
    F.col("casual_trips").desc()
).show(10, truncate=False)

+----------------+----------------------------------+-----------+------------+------------+----------------+------------------+-----------------+
|start_station_id|start_station_name                |total_trips|casual_trips|member_trips|casual_share_pct|rank_casual_volume|rank_casual_share|
+----------------+----------------------------------+-----------+------------+------------+----------------+------------------+-----------------+
|13022           |Streeter Dr & Grand Ave           |77318      |59513       |17805       |76.97           |1                 |9                |
|13300           |DuSable Lake Shore Dr & Monroe St |52493      |40829       |11664       |77.78           |2                 |8                |
|13042           |Michigan Ave & Oak St             |46293      |28851       |17442       |62.32           |3                 |37               |
|CHI01747        |Navy Pier                         |35135      |27414       |7721        |78.02           |4               

In [32]:
gold_station_ranking.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LAKE}/gold/station_ranking")

In [33]:
import inspect

print(inspect.signature(vers_postgres))
print(inspect.getsource(vers_postgres))

(df, table, mode='overwrite')
def vers_postgres(df, table, mode="overwrite"):
    """Copie un DataFrame Spark dans une table PostgreSQL (couche de restitution)."""
    df.write.jdbc(PG_URL, table, mode=mode, properties=PG_PROPS)
    print(f"Table PostgreSQL « {table} » écrite.")



In [34]:
print("PG_URL :", PG_URL)
print("PG_PROPS :", PG_PROPS)

PG_URL : jdbc:postgresql://postgres:5432/nyctaxi
PG_PROPS : {'user': 'cours', 'password': 'cours', 'driver': 'org.postgresql.Driver'}


In [35]:
PG_URL = "jdbc:postgresql://postgres:5432/divvy"

PG_PROPS = {
    "user": "divvy",
    "password": "divvy",
    "driver": "org.postgresql.Driver"
}

print("PG_URL :", PG_URL)
print("PG_PROPS :", PG_PROPS)

PG_URL : jdbc:postgresql://postgres:5432/divvy
PG_PROPS : {'user': 'divvy', 'password': 'divvy', 'driver': 'org.postgresql.Driver'}


In [36]:
gold_station_period = spark.read.format("delta").load(
    f"{LAKE}/gold/station_period"
)

gold_monthly_overview = spark.read.format("delta").load(
    f"{LAKE}/gold/monthly_overview"
)

gold_station_ranking = spark.read.format("delta").load(
    f"{LAKE}/gold/station_ranking"
)

In [38]:
import commun.spark_utils as su

su.PG_URL = PG_URL
su.PG_PROPS = PG_PROPS

print("URL :", su.PG_URL)
print("Propriétés :", su.PG_PROPS)

URL : jdbc:postgresql://postgres:5432/divvy
Propriétés : {'user': 'divvy', 'password': 'divvy', 'driver': 'org.postgresql.Driver'}


In [39]:
vers_postgres(gold_station_period, "gold_station_period")

Table PostgreSQL « gold_station_period » écrite.


In [1]:
from commun.spark_utils import *

spark = creer_session("Divvy-Silver-Gold")
spark.conf.set("spark.sql.session.timeZone", "America/Chicago")

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040


In [2]:
import commun.spark_utils as su

print(su.PG_URL)
print(su.PG_PROPS)

jdbc:postgresql://postgres:5432/divvy
{'user': 'divvy', 'password': 'divvy', 'driver': 'org.postgresql.Driver'}


In [3]:
gold_station_period = spark.read.format("delta").load(f"{LAKE}/gold/station_period")

gold_monthly_overview = spark.read.format("delta").load(f"{LAKE}/gold/monthly_overview")

gold_station_ranking = spark.read.format("delta").load(f"{LAKE}/gold/station_ranking")

In [4]:
vers_postgres(gold_monthly_overview, "gold_monthly_overview")

vers_postgres(gold_station_ranking, "gold_station_ranking")

Table PostgreSQL « gold_monthly_overview » écrite.
Table PostgreSQL « gold_station_ranking » écrite.
